In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=Hypothesis/T-tests"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except AssertionError:
      pass

# Introduction

 
The hypothesis tests that you will learn to perform using Student's t-distribution in these exercises are very similar to the ones you learned about in the previous set of exercises. Much like in the previous exercise, the __null hypothesis__ for all our tests will be that the expectation, $\mathbb{E}(X) = \mu$, for the random variables we sampled has a particular value $\mu_0$.  In other words:

$$
H_0: \mu = \mu_0
$$

We will then discuss how to do one and two-tailed hypothesis tests to test this __null hypothesis__ against the following three alternatives.

$$
H_1: \mu < \mu_0 \qquad H_1: \mu \ne \mu_0 \qquad H_1: \mu > \mu_0
$$ 

The difference between what we are doing here and what you did previously is that you do not have a value for the variance. You instead need to generate an estimate of the variance by computing the sample variance for the $n$ random variables, $X_i$ upon which the hypothesis test is being performed using:

$$
S^2 = \frac{n}{n-1} \left( \frac{1}{n} \sum_{i=1}^n X_i^2 - \overline{X}^2 \right) \qquad \textrm{where} \qquad \overline{X} = \frac{1}{n} \sum_{i=1}^n X_i
$$

We have used python to calculate this quantity in many earlier earlier exercises. However, the first exercise below, nevertheless, begins by reivising how to calculate a sample variance. Before you get on to that, though, you first need to run the cell and the top of this notebook and the cell below to load the various libraries that you will need to complete the exercies. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats

# Calculating the sample mean and sample variance

As discussed in the introduction, when we do the particular kind of hypothesis test that we are studying in this notebook we will be given a sample of $n$ independent and identical random variables.  The first step of the hypothesis test then involves computing a sample mean and sample variance from this data. We will thus begin this exercise by revising how to calculate a sample mean and a sample variance.

The cell below contains the start of a function called `sample_mean_and_var` that takes a NumPy array called `sample` in input.  This array contains a set of $n$ independent and identical random variables. Your task is to edit the `sample_mean_and_var` function so that it returns a sample mean and a sample variance computed from the data in the NumPy array `sample`.

Remember that you can calculate the sum of a NumPy array using the command `sum` and you can calculate the number of elements in the array by using `len`.  

I have loaded a sample of random variables that I generated into the array called `mysample`. Once you get your code operating correctly the sample mean and the sample variance for this data will be output.  

__The mean should be the first quantity returned by your function and the variance should be the second quantity.__

In [ ]:
def sample_mean_and_var( sample ) :
    mean, var = 0, 0
    # Your code goes here
    
    return mean, var

# You do not need to modify any of the code from here onwards
mysample = np.loadtxt( "https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/Hypothesis/Basics/sample_data.dat" )
mymean, myvar = sample_mean_and_var( mysample )
print( "The mean for the data in mysample is", mymean, "and the variance is", myvar )

In [ ]:
runtest(["test_ex1"])

# Calculating the test statistic

In the previous exercise you computed:

$$
\overline{X} = \frac{1}{n} \sum_{i=1}^n X_i \qquad \textrm{and} \qquad S^2 = \frac{n}{n-1} \left( \frac{1}{n} \sum_{i=1}^n X_i^2 - \overline{X}^2 \right) 
$$

from the sample of $n$ random variables, $X_i$, that were provided in the input file.  $\overline{X}$ is an estimator for the expectation (or population mean), $\mu$, of the distribution from which $X_i$ is a sample.  Similarly, $S^2$ is an estimator for the variance for the variance of this distribution. 

We compute the test statistic, $T$, for the hypothesis test we are studying here using:

$$
T = \frac{\overline{X} - \mu_0}{\sqrt{S^2/n}}
$$

where $\mu_0$ is the valuse for the expectation of the distribution that the $X_i$ are sampling from if the null hypothesis is true.  

Your task for this exercise is to complete the function called `teststat` that I have started below for computing this test statistic.  You will notice that this function takes two arguments:

- `sample` is a NumPy array that contains the sample of random variables upon which the hypothesis test is being performed.
- `mu0` is the value for the expectation of the distribution that the $n$ $X_i$ values in the sample are taken from under the assumption that the null hypothesis is true.

You will notice that I have used the function that you wrote in the last exerice to compute $\overline{X}$ and $S^2$ from the data in `sample` for you.

In [ ]:
def teststat( sample, mu0 ) : 
    xbar, S2 = sample_mean_and_var( sample )
    # Your code goes here
    
    return 1

# You do not need to modify any of the code from here onwards
mysample = np.loadtxt( "https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/Hypothesis/Basics/sample_data.dat" )
print("The test statistic for a test to determine whether mu=0 for the data in sample_data.dat is", teststat( mysample, 0) )

In [ ]:
runtest(["test_ex2"])

# Calculating the p-value

The factor of $S^2/n$ that appears square rooted in the denominator of the expression for the test statistic that you have computed is an estimator for the variance of $\overline{X}$. Consequently, if $S^2$ was equal to the true (population) variance of the distribution that the $X_i$ were sampled from and if the __null hypothesis__ was true, then the test statistic, $T$, that you have just computed would be a standard normal random variable.  However, $S^2$ is an estimator for the variance and $T$ is thus a sample from a Student's t-distribution with $(n-1)$ degrees of freedom.  Calculating the value of the cumulative probability density function for this function at $x$ using python can be done as follows:

````
y = scipy.stats.t.cdf(x,n-1)
````

where the variance $n$ here is the size of the sample of random variables.

Given this information your task for this exercise is to complete the three functions below. Each of these functions takes the following two variables in input:

* `sample` - the sample of independent and identicaly random variables on which we are performing the hypothesis test
* `mu0` - the value for the expectation of the random variables that is assumed under the null hypothesis.

You will also see that I have called the function for calculating the test statistic, `teststat`, that you wrote in an earlier exercise within each of them.  Your task is thus to convert the value of the __test statistic__, `t`, that is returned by this function to a __p-value__ for each type of test. For the three functions the __null, $H_0$ and alternative hypothesis, $H_1$,__ are:

* `pval_lower` - $H_0: \mu=\mu_0$ against $H_1: \mu<\mu_0$
* `pval_not` - $H_0: \mu=\mu_0$ against $H_1: \mu\ne \mu_0$
* `pval_higher`-  $H_0: \mu=\mu_0$ against $H_1: \mu>\mu_0$

Where $\mu_0$ is the value of the input variable `mu0`.  

I have included code at the end of the cell below to perform these hypothesis tests with $\mu_0=0$ on the data set that we have analysed in earlier exercises.  

In [ ]:
def pval_lower( sample, mu0 ) : 
    t = teststat( sample, mu0 )
    # Your code goes here


def pval_not( sample, mu0 ) : 
    t = teststat( sample, mu0 )
    # Your code goes here


def pval_higher( sample, mu0 ) : 
    t = teststat( sample, mu0 )
    # Your code goes here


# You do not need to modify any of the code from here onwards
# Notice that I don't really need to load the data here again
# as I read the file sample_data.dat earlier in the notebook
# and set mysample equal to its contents. I am only reading this file here as I have no way of 
# knowing whether you changed the value of the variable mysample
# between the earlier cell where I read it and here.
mysample = np.loadtxt( "https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/Hypothesis/Basics/sample_data.dat" )
print( "The p-value for a test of H_0: mu=0 against H_1 mu<0", pval_lower( mysample, 0 ) )
print( "The p-value for a test of H_0: mu=0 against H_1 mu \ne 0", pval_not( mysample, 0 ) )
print( "The p-value for a test of H_0: mu=0 against H_1 mu>0", pval_higher( mysample, 0 ) )

In [ ]:
runtest(["test_ex3a","test_ex3b","test_ex3c"])

# Conclusions

You now have all the information you should need to reproduce most of the figures in the example report. To construct figure 1 you simply generate multiple samples of $n$ random variables. You then calculate the test statistic using the code from the second of the exercises above for each of these samples. The values for the __test statistics__ are random variabels so you can compute summary statstics from them.  These forms of analysis were covered in the exercises on random variables and you can use what you learend there to compute the means, confidence limits and histograms (with error bars) that are shown in figure 1 in the example report.

To plot the probability density functions that appear in the second figure of the example report you can use the following python code:

````
x = np.linspace( -4, 4, 1000 )
y = scipy.stats.t.pdf( x, 10 )

plt.plot( x, y, 'k-' )
plt.xlabel("Random variable")
plt.ylabel("Probability density")
plt.show()
````

This will produce a plot of the probability density function for a Student's t-distribution with 10 degrees of freedom. If you use `y = scipy.stats.norm.pdf( x )` you can plot the normal distributions instead of the t-distribution.

Note that you could plot the logarithm of the probability density by using `y = np.log( scipy.stats.t.pdf( x, 10 ) )`. However, I instead used commands the same commands as above when plotting figure 2 but added the command `plt.yscale("log")` to convert the y-axis to a logarithmic scale.

Lastly, note that I have not shown you how to construct figure 3 because (a) I don't think that it is easy for find a way to reuse this analysis in other reports and (b) it shouldn't be too hard for you to work out how to reproduce that grpah yourself using the information in the example report.